[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/s3_raytracer/s3_raytracer.ipynb)

# Ray Tracing on the 3-Sphere

This example renders four identical ellipsoids in a closed, curved space, the 3-sphere, as seen by an eye walking towards them. Light there runs along great circles, so the ellipsoids lined up along the line of sight do not keep shrinking with distance: past a quarter turn the great circles from the eye converge again towards the eye's antipode, and the farthest ellipsoid looms larger than the middle ones.

The 3-sphere is modelled in the geometric algebra of four-dimensional space, where points of the sphere are antivectors and great spheres, its planes, are vectors, and an ellipsoid is a dual quadric, a map from planes to points, placed by a motor. Seen from the eye, a body's outline is its quadric pushed through the central projection from the eye onto the image sphere a quarter turn ahead: a conic in the pixel chart, negative inside the outline. With the eye's polar plane under the body, the conic also gives each pixel's depth along its great circle, and the nearest body wins.

In the matrix notation of projective geometry the projection of a quadric $Q$ from a point $\mathbf e$ reads as the cone $(\mathbf e^{\mathsf T} Q\,\mathbf e)\,Q - Q\mathbf e\,\mathbf e^{\mathsf T}Q$, cut by the image plane to the outline conic.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from examples.animation import save_figure, show_animation
from examples.quadrics.s3_raytracer import render
from examples.quadrics.s3_raytracer.core import (
    DualQuadric, Motor, Plane, Point, Quadric, Scalar, ScreenConic, ScreenPoint, ScreenPolar, direction, mv, origin, pixel_chart,
)

np.set_printoptions(precision=4, suppress=True)

## 1. Four identical ellipsoids along the line of sight

An ellipsoid at the origin is the dual quadric of its three axis directions, points a quarter turn away, each paired with itself and weighted by the squared tangent of its angular half-width, less the origin paired with itself. Four copies are carried by motors to increasing angular distances along the line of sight, a little to the side and up or down. Each body is drawn by its primal form, the inverse, a map from points to their polar planes.

In [ ]:
# The ellipsoid's angular half-widths; each copy's distance along the line of sight, and its offsets
# sideways and upward, by angle; its turn; its colour; the light's direction; the field of view.
half_widths = np.array([0.2, 0.28, 0.15])
distances = np.array([0.7, 1.4, 2.1, 2.6])
sideways, upward = np.array([-0.4, 0.4, -0.4, 0.4]), np.array([-0.3, -0.3, 0.3, 0.3])
turn = 0.8
colours = np.array([[0.9, 0.3, 0.3], [0.3, 0.8, 0.4], [0.3, 0.5, 0.95], [0.95, 0.8, 0.3]])
light = direction(np.array([-0.4, 0.6, 0.7]))
field_of_view = np.radians(80.0)

In [ ]:
axes = direction(np.eye(3))                                                    # [3] Point
ellipsoid = (axes * (axes & Plane) * np.tan(half_widths) ** 2).sum(axis=0) - origin * (origin & Plane)   # [] Point <- Plane
placed = ((mv.xw * (distances / 2)).exp() * (mv.yw * (sideways / 2)).exp() * (mv.zw * (upward / 2)).exp()
          * (mv.xy * (turn / 2)).exp())                                        # [bodies] Motor
bodies = placed >> ellipsoid(placed << Plane)                                  # [bodies] Point <- Plane
surfaces = bodies.inverse()                                                    # [bodies] Plane <- Point

## 2. Outlines and depths by projection

From the eye, the cone of rays tangent to a body is the eye's polar plane squared, less the body's form scaled by the eye's own value under it: non-negative on the rays that meet the body. Pulled into the pixel chart, the projection gives each body a screen conic and a linear form, the eye's polar; a pixel's depth is the larger root of the quadratic they make, the cotangent of the angle travelled along the pixel's great circle. Where a pixel's great circle meets a body exactly when the conic admits a root, the 2D test agrees with the full one.

In [ ]:
def pixel_chart(fov: float, shape: tuple[int, int]) -> ScreenPoint:
    """Homogeneous screen points (1, u, v) of a pinhole grid looking along +x from the origin."""
    v, u = np.meshgrid(np.linspace(1.0, -1.0, shape[0]) * shape[0] / shape[1], np.linspace(-1.0, 1.0, shape[1]), indexing="ij")
    axes = mv(ScreenPoint, np.eye(3))                                          # [3] ScreenPoint
    return axes[0] + axes[1] * (np.tan(fov / 2) * u.ravel()) + axes[2] * (np.tan(fov / 2) * v.ravel())


def outlines(eye_frame: Motor, surfaces: Quadric) -> Quadric:
    """The cone of rays from the eye tangent to each body: the eye's polar plane squared, less the
    form scaled by the eye's own value. Non-negative on the directions whose ray meets the body."""
    eye = eye_frame >> origin
    polar = surfaces(eye)
    return polar * (Point & polar) - surfaces * (eye & polar)


def inside(cone: Quadric, rays: ScreenPoint) -> np.ndarray:
    """Hit test of ray directions against the bodies' outline cones."""
    return (rays & cone.reshape(-1, 1)(rays)) >= 0.0


def project(eye_frame: Motor, surfaces: Quadric) -> tuple[ScreenConic, ScreenPolar]:
    """Project each quadric to its screen conic and eye-polar linear form; the eye must be off-surface."""
    eye = eye_frame >> origin
    screen = eye_frame >> ScreenPoint
    normalized = surfaces / (eye & surfaces(eye))
    polar = screen & normalized(eye)
    return (screen & normalized(screen)) - polar * polar, polar


def reproject(conic: ScreenConic, polar: ScreenPolar, pixels: ScreenPoint) -> Scalar:
    """First-hit screen depth, proportional to the cotangent of the angle: larger is nearer; NaN
    means a miss."""
    with np.errstate(invalid="ignore"):
        return -polar(pixels) + (-conic(pixels, pixels)).square_root()


# From the origin: the coarse chart, which rays meet which body, and the full test along each great circle.
chart = pixel_chart(field_of_view, (90, 120))                                  # [pixels] ScreenPoint
covered = inside(outlines(mv.rotor(), surfaces), chart)                        # [bodies, pixels]
at_eye, along = surfaces.reshape(-1, 1)(origin), surfaces.reshape(-1, 1)(chart)   # [bodies, 1], [bodies, pixels] Plane
# Along the great circle `origin + chart * t` the body's form is `a * t**2 + 2 * b * t + c`.
a, b, c = chart & along, chart & at_eye, origin & at_eye                       # [bodies, pixels] Scalar each
discriminant = b * b - a * c                                                   # [bodies, pixels] Scalar
conics, polars = project(mv.rotor(), surfaces)                                 # [bodies] ScreenConic, ScreenPolar

In [ ]:
real = discriminant.to_array()
clear = np.abs(real) > 1e-3 * np.abs(real).max(axis=1, keepdims=True)
print("pixels covered per body:", covered.sum(axis=1))
print("outline test agrees with the great circles away from the outlines:", np.array_equal(covered[clear], (real >= 0.0)[clear]))

## 3. Walking towards them

The eye walks along the line of sight; the scene stays where it is. Each frame keeps, per pixel, the body with the largest depth, reconstructs only that hit on the sphere, and shades it by a point light on the sphere itself, with the falloff of a point source along great circles. As the eye walks, the nearest ellipsoid grows, while the farthest, already past a quarter turn, was large from the start.

In [ ]:
# How far the eye walks, in frames; the image's pixels and supersampling; a frame's duration in milliseconds.
walk, n_frames = 1.2, 48
shape, supersample, frame_duration = (360, 480), 3, 80

In [ ]:
eye_frames = (mv.xw * (np.linspace(0.0, walk, n_frames, endpoint=False) / 2)).exp()   # [frames] Motor
save_figure(render.draw_walk(eye_frames, surfaces, colours, light, field_of_view, shape, supersample), "s3_raytracer")
show_animation(render.frames(eye_frames, surfaces, colours, light, field_of_view, shape, supersample), "s3_raytracer", frame_duration)

In [ ]:
# checks
# The outline test agrees with the great circles away from the outlines; the screen conic is the
# discriminant over c squared; reprojected hits lie on their surfaces.
assert np.array_equal(covered[clear], (real >= 0.0)[clear])
np.testing.assert_allclose(conics.reshape(-1, 1)(chart, chart).to_array(), -(discriminant / (c * c)).to_array(), atol=1e-8)
for body in range(len(distances)):
    pixels = np.nonzero(covered[body])[0][::10]
    hit = (origin * reproject(conics[body], polars[body], chart[pixels]) + chart[pixels]).normalized()
    np.testing.assert_allclose((hit & surfaces[body](hit)).to_array(), 0.0, atol=1e-8)